<a href="https://colab.research.google.com/github/TottiPuc/Machine_learning/blob/master/NYC_Taxi_Trip_Duration_Regression_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYC Taxi Trip Duration Prediction



### Business problem
Ride-hailing platforms need to estimate how long a trip will take before the ride is completed. A reliable trip-duration estimate can improve driver dispatching, customer experience, route planning, and fleet utilization.

### Objective
Build a supervised machine-learning regression model that predicts **trip duration in seconds** using information available at or before pickup time.

### Main steps
1. Load and inspect the dataset.
2. Perform data-quality checks.
3. Explore the target and important variables.
4. Detect and remove unrealistic observations.
5. Prevent target leakage.
6. Engineer useful temporal and geographic features.
7. Build a baseline model.
8. Train Linear Regression and Ridge Regression.
9. Train an improved nonlinear model.
10. Compare models using MAE, RMSE, R², and RMSLE.
11. Analyze residuals and feature importance.
12. Save the best model.

> **Important:** `dropoff_datetime` is not used as a predictor because it directly reveals the target duration and would create target leakage.

## 1. Import libraries

All libraries used below are available in Google Colab by default.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.inspection import permutation_importance
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 2. Load the dataset

The notebook first looks for the CSV in common locations. If the file is not found and the notebook is running in Google Colab, it will ask you to upload it.

In [ ]:
FILE_NAME = "nyc_taxi_trip_duration.csv"

candidate_paths = [
    f"/content/{FILE_NAME}",
    FILE_NAME,
    f"/mnt/data/{FILE_NAME}"
]

DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print(f"Please upload: {FILE_NAME}")
        uploaded = files.upload()
        if FILE_NAME in uploaded:
            DATA_PATH = FILE_NAME
        else:
            csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
            if not csv_files:
                raise FileNotFoundError("No CSV file was uploaded.")
            DATA_PATH = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {FILE_NAME}. Place it in the working directory."
        )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 3. Understand the dataset structure

The first inspection checks column names, data types, and basic descriptive statistics.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()

In [ ]:
display(df.describe(include="all").T)

### Data dictionary

| Column | Description |
|---|---|
| `id` | Unique trip identifier |
| `vendor_id` | Taxi / vendor identifier |
| `pickup_datetime` | Pickup date and time |
| `dropoff_datetime` | Drop-off date and time |
| `passenger_count` | Number of passengers |
| `pickup_longitude` | Pickup longitude |
| `pickup_latitude` | Pickup latitude |
| `dropoff_longitude` | Drop-off longitude |
| `dropoff_latitude` | Drop-off latitude |
| `store_and_fwd_flag` | Whether the trip record was temporarily stored before being sent |
| `trip_duration` | **Target variable:** trip duration in seconds |

## 4. Data-quality checks

We check missing values, duplicate rows, impossible passenger counts, extreme trip durations, and suspicious coordinates.

In [ ]:
quality_report = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100,
    "unique_values": df.nunique()
})

display(quality_report)

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Passenger count distribution:")
display(df["passenger_count"].value_counts().sort_index().to_frame("count"))

print("\nStore-and-forward flag distribution:")
display(df["store_and_fwd_flag"].value_counts(dropna=False).to_frame("count"))

print("\nTrip duration summary:")
display(
    df["trip_duration"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999])
    .to_frame()
)

## 5. Convert date columns

The original date columns are strings. Converting them to `datetime` allows us to extract hour, weekday, month, and other time-related information.

In [ ]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

print("Pickup date range:")
print(df["pickup_datetime"].min(), "to", df["pickup_datetime"].max())

print("\nDrop-off date range:")
print(df["dropoff_datetime"].min(), "to", df["dropoff_datetime"].max())

## 6. Exploratory Data Analysis (EDA)

### 6.1 Distribution of trip duration

The raw target is strongly right-skewed because a small number of trips have extremely large durations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["trip_duration"], bins=100)
ax.set_title("Raw Trip Duration Distribution")
ax.set_xlabel("Trip Duration (seconds)")
ax.set_ylabel("Frequency")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log1p(df["trip_duration"]), bins=100)
ax.set_title("Log-Transformed Trip Duration Distribution")
ax.set_xlabel("log(1 + Trip Duration)")
ax.set_ylabel("Frequency")
plt.show()

### 6.2 Trip duration by pickup hour

This helps us understand whether traffic patterns may influence travel time.

In [ ]:
eda_time = df.copy()
eda_time["pickup_hour"] = eda_time["pickup_datetime"].dt.hour

hourly_duration = (
    eda_time.groupby("pickup_hour")["trip_duration"]
    .median()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly_duration.index, hourly_duration.values, marker="o")
ax.set_title("Median Trip Duration by Pickup Hour")
ax.set_xlabel("Pickup Hour")
ax.set_ylabel("Median Trip Duration (seconds)")
ax.set_xticks(range(24))
plt.show()

del eda_time

### 6.3 Pickup locations

A random sample is used so the plot remains fast and readable.

In [ ]:
geo_sample = df.sample(min(30000, len(df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    geo_sample["pickup_longitude"],
    geo_sample["pickup_latitude"],
    s=3,
    alpha=0.25
)
ax.set_title("Sample of Pickup Locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 7. Prevent target leakage

`trip_duration` can be reconstructed from:

`dropoff_datetime - pickup_datetime`

Therefore, using `dropoff_datetime` in the model would make the prediction unrealistically easy and would not represent the real business scenario, because the trip has not ended yet when the prediction must be made.

We will **not** use:
- `id` because it is only an identifier.
- `dropoff_datetime` because it leaks the target.

In [ ]:
duration_from_dates = (
    df["dropoff_datetime"] - df["pickup_datetime"]
).dt.total_seconds()

difference = (duration_from_dates - df["trip_duration"]).abs()

print("Maximum absolute difference between calculated duration and target:")
print(difference.max(), "seconds")

## 8. Clean unrealistic observations

The dataset has no missing values, but it contains extreme outliers and a few invalid observations.

We apply broad business rules:
- Trip duration must be between **60 seconds and 2 hours**.
- Passenger count must be between **1 and 6**.
- Pickup and drop-off coordinates must be inside a broad New York City area.

These rules remove obvious anomalies while preserving the great majority of valid trips.

In [ ]:
original_rows = len(df)

clean_mask = (
    df["trip_duration"].between(60, 7200)
    & df["passenger_count"].between(1, 6)
    & df["pickup_longitude"].between(-75, -72)
    & df["dropoff_longitude"].between(-75, -72)
    & df["pickup_latitude"].between(40, 42)
    & df["dropoff_latitude"].between(40, 42)
)

df_clean = df.loc[clean_mask].copy()

removed_rows = original_rows - len(df_clean)
removed_percent = removed_rows / original_rows * 100

print(f"Original rows: {original_rows:,}")
print(f"Rows after cleaning: {len(df_clean):,}")
print(f"Rows removed: {removed_rows:,} ({removed_percent:.2f}%)")

In [ ]:
display(
    df_clean["trip_duration"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_frame()
)

## 9. Feature engineering

Trip duration is affected by both **distance** and **traffic-related time patterns**.

We create:

### Geographic features
- Haversine distance: straight-line distance over the Earth's surface.
- Manhattan-like distance: approximate horizontal plus vertical geographic movement.

### Temporal features
- Pickup hour.
- Day of week.
- Month.
- Day of month.
- Weekend indicator.
- Rush-hour indicator.
- Night-trip indicator.
- Cyclical encodings for hour and weekday.

Cyclical encoding is useful because hour 23 and hour 0 are close in time even though their numeric values are far apart.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def manhattan_distance(lat1, lon1, lat2, lon2):
    horizontal = haversine_distance(lat1, lon1, lat1, lon2)
    vertical = haversine_distance(lat1, lon1, lat2, lon2)
    return horizontal + vertical

In [ ]:
df_model = df_clean.copy()

# Geographic features
df_model["distance_km"] = haversine_distance(
    df_model["pickup_latitude"],
    df_model["pickup_longitude"],
    df_model["dropoff_latitude"],
    df_model["dropoff_longitude"]
)

df_model["manhattan_distance_km"] = manhattan_distance(
    df_model["pickup_latitude"],
    df_model["pickup_longitude"],
    df_model["dropoff_latitude"],
    df_model["dropoff_longitude"]
)

# Time features
df_model["pickup_hour"] = df_model["pickup_datetime"].dt.hour
df_model["pickup_dayofweek"] = df_model["pickup_datetime"].dt.dayofweek
df_model["pickup_month"] = df_model["pickup_datetime"].dt.month
df_model["pickup_day"] = df_model["pickup_datetime"].dt.day

df_model["is_weekend"] = (
    df_model["pickup_dayofweek"] >= 5
).astype(int)

df_model["rush_hour"] = (
    df_model["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19])
).astype(int)

df_model["night_trip"] = (
    df_model["pickup_hour"].isin([0, 1, 2, 3, 4, 5])
).astype(int)

# Cyclical encodings
df_model["hour_sin"] = np.sin(
    2 * np.pi * df_model["pickup_hour"] / 24
)
df_model["hour_cos"] = np.cos(
    2 * np.pi * df_model["pickup_hour"] / 24
)

df_model["dow_sin"] = np.sin(
    2 * np.pi * df_model["pickup_dayofweek"] / 7
)
df_model["dow_cos"] = np.cos(
    2 * np.pi * df_model["pickup_dayofweek"] / 7
)

# Binary encoding
df_model["store_and_fwd_flag_binary"] = (
    df_model["store_and_fwd_flag"] == "Y"
).astype(int)

display(df_model.head())

### 9.1 Relationship between distance and trip duration

A positive relationship is expected, although traffic and route conditions create substantial variability.

In [ ]:
plot_sample = df_model.sample(
    min(30000, len(df_model)),
    random_state=RANDOM_STATE
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    plot_sample["distance_km"],
    plot_sample["trip_duration"],
    s=5,
    alpha=0.2
)
ax.set_title("Trip Duration vs Haversine Distance")
ax.set_xlabel("Haversine Distance (km)")
ax.set_ylabel("Trip Duration (seconds)")
ax.set_xlim(0, plot_sample["distance_km"].quantile(0.99))
plt.show()

## 10. Select predictors and target

Only variables that would realistically be available when the trip begins are included.

In [ ]:
feature_columns = [
    "vendor_id",
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "store_and_fwd_flag_binary",
    "distance_km",
    "manhattan_distance_km",
    "pickup_month",
    "pickup_day",
    "pickup_hour",
    "pickup_dayofweek",
    "is_weekend",
    "rush_hour",
    "night_trip",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos"
]

X = df_model[feature_columns].copy()
y = df_model["trip_duration"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())

## 11. Train-test split

We use:
- **80%** of the data for training.
- **20%** for testing.
- A fixed random seed for reproducibility.

For a production system, a time-based validation strategy would be even more realistic because future trips should be predicted using past trips.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", f"{len(X_train):,}")
print("Testing rows:", f"{len(X_test):,}")

## 12. Evaluation metrics

We evaluate the models with:

- **MAE (Mean Absolute Error):** average absolute prediction error in seconds.
- **RMSE (Root Mean Squared Error):** penalizes large errors more strongly.
- **R²:** proportion of target variance explained by the model.
- **RMSLE:** evaluates relative prediction error on a logarithmic scale.

Lower MAE, RMSE, and RMSLE are better. Higher R² is better.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_pred_nonnegative = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_true, y_pred_nonnegative)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_nonnegative))
    r2 = r2_score(y_true, y_pred_nonnegative)

    rmsle = np.sqrt(
        mean_squared_error(
            np.log1p(y_true),
            np.log1p(y_pred_nonnegative)
        )
    )

    return {
        "MAE_seconds": mae,
        "RMSE_seconds": rmse,
        "R2": r2,
        "RMSLE": rmsle
    }

## 13. Baseline model

A machine-learning model should outperform a simple reference strategy.

The baseline predicts the **median trip duration from the training set** for every test observation.

In [ ]:
baseline_value = y_train.median()
baseline_predictions = np.full(len(y_test), baseline_value)

results = {}

results["Median Baseline"] = regression_metrics(
    y_test,
    baseline_predictions
)

print("Baseline prediction:", baseline_value, "seconds")
display(pd.DataFrame(results).T)

## 14. Linear Regression

Linear Regression is the natural first regression model for this assignment.

The numeric variables are standardized inside a pipeline.

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

results["Linear Regression"] = regression_metrics(
    y_test,
    linear_predictions
)

display(pd.DataFrame(results).T.sort_values("RMSE_seconds"))

## 15. Ridge Regression

Ridge Regression adds L2 regularization. This is useful when predictors are correlated, such as straight-line distance, Manhattan-like distance, and the raw coordinates.

In [ ]:
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_test)

results["Ridge Regression"] = regression_metrics(
    y_test,
    ridge_predictions
)

display(pd.DataFrame(results).T.sort_values("RMSE_seconds"))

## 16. Improved nonlinear model: Histogram Gradient Boosting

Linear models assume a mostly linear relationship between predictors and trip duration. In reality, travel time depends on nonlinear interactions between distance, location, pickup hour, weekday, and traffic patterns.

`HistGradientBoostingRegressor` is:
- Fast on large datasets.
- Available directly in scikit-learn.
- Able to model nonlinear relationships and feature interactions.

In [ ]:
hgb_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=63,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

hgb_model.fit(X_train, y_train)
hgb_predictions = hgb_model.predict(X_test)

results["HistGradientBoosting"] = regression_metrics(
    y_test,
    hgb_predictions
)

results_df = (
    pd.DataFrame(results)
    .T
    .sort_values("RMSE_seconds")
)

display(results_df)

## 17. Compare model performance

The best model should produce:
- The lowest MAE.
- The lowest RMSE.
- The lowest RMSLE.
- The highest R².

In [ ]:
comparison = results_df.copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(comparison.index, comparison["MAE_seconds"])
ax.set_title("Model Comparison — MAE")
ax.set_ylabel("MAE (seconds)")
ax.tick_params(axis="x", rotation=20)
plt.show()

display(comparison)

## 18. Select the best model

The model with the lowest test RMSE is selected automatically.

In [ ]:
model_objects = {
    "Median Baseline": None,
    "Linear Regression": linear_model,
    "Ridge Regression": ridge_model,
    "HistGradientBoosting": hgb_model
}

best_model_name = results_df.index[0]
best_model = model_objects[best_model_name]

print("Best model:", best_model_name)
display(results_df.loc[[best_model_name]])

## 19. Actual vs predicted values

A good model should place points reasonably close to the diagonal reference line.

In [ ]:
if best_model_name == "HistGradientBoosting":
    best_predictions = hgb_predictions
elif best_model_name == "Linear Regression":
    best_predictions = linear_predictions
elif best_model_name == "Ridge Regression":
    best_predictions = ridge_predictions
else:
    best_predictions = baseline_predictions

rng = np.random.RandomState(RANDOM_STATE)
sample_size = min(10000, len(y_test))
sample_idx = rng.choice(len(y_test), size=sample_size, replace=False)

y_test_array = y_test.to_numpy()
pred_array = np.asarray(best_predictions)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    y_test_array[sample_idx],
    pred_array[sample_idx],
    s=6,
    alpha=0.25
)

limit = np.percentile(y_test_array[sample_idx], 99)
ax.plot([0, limit], [0, limit], linestyle="--")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_title(f"Actual vs Predicted — {best_model_name}")
ax.set_xlabel("Actual Trip Duration (seconds)")
ax.set_ylabel("Predicted Trip Duration (seconds)")
plt.show()

## 20. Residual analysis

Residuals are defined as:

`actual duration - predicted duration`

Ideally, residuals should be centered around zero without a strong systematic pattern.

In [ ]:
residuals = y_test_array - pred_array

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(residuals, bins=100)
ax.set_title(f"Residual Distribution — {best_model_name}")
ax.set_xlabel("Residual (seconds)")
ax.set_ylabel("Frequency")
ax.set_xlim(
    np.percentile(residuals, 1),
    np.percentile(residuals, 99)
)
plt.show()

print("Mean residual:", residuals.mean())
print("Median residual:", np.median(residuals))

## 21. Permutation feature importance

Permutation importance measures how much model performance deteriorates when one feature is randomly shuffled.

To keep the calculation fast, we use a random sample from the test set.

In [ ]:
if best_model is not None:
    importance_sample = X_test.sample(
        min(15000, len(X_test)),
        random_state=RANDOM_STATE
    )
    importance_target = y_test.loc[importance_sample.index]

    perm = permutation_importance(
        best_model,
        importance_sample,
        importance_target,
        n_repeats=3,
        random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error"
    )

    importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": perm.importances_mean
    }).sort_values("importance", ascending=False)

    display(importance_df)

    top_features = importance_df.head(12).sort_values("importance")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_features["feature"], top_features["importance"])
    ax.set_title(f"Permutation Feature Importance — {best_model_name}")
    ax.set_xlabel("Increase in MAE when feature is shuffled")
    plt.show()

## 22. Inspect prediction examples

This table compares real trip durations with model predictions and absolute errors.

In [ ]:
prediction_examples = X_test.copy()
prediction_examples["actual_duration_seconds"] = y_test
prediction_examples["predicted_duration_seconds"] = np.clip(
    best_predictions,
    0,
    None
)
prediction_examples["absolute_error_seconds"] = (
    prediction_examples["actual_duration_seconds"]
    - prediction_examples["predicted_duration_seconds"]
).abs()

display(
    prediction_examples[
        [
            "distance_km",
            "pickup_hour",
            "pickup_dayofweek",
            "actual_duration_seconds",
            "predicted_duration_seconds",
            "absolute_error_seconds"
        ]
    ]
    .sample(10, random_state=RANDOM_STATE)
    .round(2)
)

## 23. Save the best model

The trained model and the ordered feature list are stored together so they can be reused later.

In [ ]:
if best_model is not None:
    artifact = {
        "model": best_model,
        "feature_columns": feature_columns
    }

    MODEL_FILE = "nyc_taxi_trip_duration_best_model.joblib"
    joblib.dump(artifact, MODEL_FILE)

    print(f"Saved model artifact: {MODEL_FILE}")
else:
    print("The baseline was selected, so no machine-learning model was saved.")

## 24. Final conclusions

This project developed a complete regression workflow for predicting NYC taxi trip duration.

### Main findings
- The dataset contains a very large number of valid trips and no missing values.
- A small number of observations contain unrealistic trip durations, passenger counts, or geographic coordinates.
- `dropoff_datetime` must be excluded because it causes target leakage.
- Geographic distance is one of the strongest predictors of trip duration.
- Pickup time also matters because traffic conditions vary during the day and across the week.
- Linear Regression provides a useful interpretable benchmark.
- Ridge Regression produces similar behavior while controlling coefficient magnitude.
- Histogram Gradient Boosting generally performs substantially better because the relationship between trip characteristics and travel time is nonlinear.

### Business interpretation
A production trip-duration model could help a ride-hailing company:
- Estimate driver availability more accurately.
- Improve pickup assignment decisions.
- Provide customers with better estimated arrival times.
- Reduce idle time and improve fleet utilization.

### Recommended future improvements
1. Add real-time traffic information.
2. Add weather conditions.
3. Use road-network distance instead of only geometric distance.
4. Add airport and borough indicators.
5. Use time-based validation to simulate future deployment.
6. Tune gradient-boosting hyperparameters systematically.
7. Monitor prediction error after deployment because traffic patterns can change over time.

## 25. Submission checklist

Before submitting the notebook:

- Run **Runtime → Run all** in Google Colab.
- Confirm that every cell executes without errors.
- Check that the model-comparison table is visible.
- Verify that the final metrics are reported.
- Make sure your name or student information is included if required by your course.
- Download the final notebook as `.ipynb`.